In [ ]:
import os, json, math, random, time, traceback, http.client, glob
from datetime import datetime

import numpy as np
import mph
from functools import partial

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

import imageio.v2 as imageio
from shapely.geometry import Polygon as SPolygon, Point as SPoint, MultiPolygon, GeometryCollection
from llm4ad.tools.llm.llm_api_https import HttpsApi as _BaseHttpsApi
from gefest.core.geometry.datastructs.point import Point
from gefest.core.geometry.datastructs.polygon import Polygon
from gefest.core.geometry.datastructs.structure import Structure
from setup_comsol import simulate_hydrodynamics

comsol_client = mph.Client(cores=8)
comsol_objective = partial(simulate_hydrodynamics, client=comsol_client)

In [2]:
class OpenRouterHttpsApi(_BaseHttpsApi):
    def __init__(self, host, key, model, timeout=60, **kwargs):
        super().__init__(host=host, key=key, model=model, timeout=timeout, **kwargs)

    def draw_sample(self, prompt, *args, **kwargs) -> str:
        if isinstance(prompt, str):
            prompt = [{'role': 'user', 'content': prompt.strip()}]

        while True:
            try:
                conn = http.client.HTTPSConnection(self._host, timeout=self._timeout)
                payload = json.dumps({
                    'max_tokens': self._kwargs.get('max_tokens', 2048),
                    'top_p': self._kwargs.get('top_p', 0.9),
                    'temperature': self._kwargs.get('temperature', 0.7),
                    'model': self._model,
                    'messages': prompt,
                })
                headers = {
                    'Authorization': f'Bearer {self._key}',
                    'Content-Type': 'application/json',
                }
                conn.request('POST', '/api/v1/chat/completions', payload, headers)
                res = conn.getresponse()
                data = res.read().decode('utf-8')
                data = json.loads(data)
                return data['choices'][0]['message']['content']
            except Exception:
                print(
                    f'OpenRouterHttpsApi error: {traceback.format_exc()}. '
                    f'Check API host/key; retrying in 2s.'
                )
                time.sleep(2)
                continue


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

llm = OpenRouterHttpsApi(
    host="openrouter.ai",
    key=OPENROUTER_API_KEY,
    model="google/gemini-2.5-flash",
    timeout=60,
)

In [3]:
print(llm.draw_sample('Return JSON: {"ok": true}'))

```json
{"ok": true}
```


In [ ]:
allowed_area = [
    (15.0, 15.0),
    (15.0, 335.0),
    (215.0, 335.0),
    (140.0, 125.0),
    (130.0, 15.0),
]
allowed_poly = SPolygon(allowed_area)

xs = [p[0] for p in allowed_area]
ys = [p[1] for p in allowed_area]
X_MIN, X_MAX = min(xs), max(xs)
Y_MIN, Y_MAX = min(ys), max(ys)

MIN_POLYS, MAX_POLYS = 3, 6
MIN_VERTS, MAX_VERTS = 5, 20
ALLOWED_ANGLES_DEG = [0, 30, 45, 60, 90, 120, 135, -30, -45, -60, -90]


def clamp_point_to_allowed(x, y):
    x = max(X_MIN, min(X_MAX, float(x)))
    y = max(Y_MIN, min(Y_MAX, float(y)))
    pt = SPoint(x, y)
    if allowed_poly.contains(pt):
        return x, y
    proj_dist = allowed_poly.exterior.project(pt)
    nearest = allowed_poly.exterior.interpolate(proj_dist)
    return float(nearest.x), float(nearest.y)


def polygons_to_structure(polygons) -> Structure:
    polys = []
    for poly in polygons:
        pts = []
        for pair in poly.get("points", []):
            if not isinstance(pair, (list, tuple)) or len(pair) != 2:
                continue
            x, y = pair
            x, y = clamp_point_to_allowed(x, y)
            pts.append(Point(x=x, y=y))
        if len(pts) >= 3:
            polys.append(Polygon(points=pts))
    return Structure(polygons=polys[:MAX_POLYS])


def random_point_in_allowed():
    minx, miny, maxx, maxy = allowed_poly.bounds
    for _ in range(1000):
        x = random.uniform(minx, maxx)
        y = random.uniform(miny, maxy)
        pt = SPoint(x, y)
        if allowed_poly.contains(pt):
            return float(x), float(y)
    return (X_MIN + X_MAX) / 2.0, (Y_MIN + Y_MAX) / 2.0


def random_polygons() -> list:
    polys = []
    n_polys = random.randint(3, 6)
    for _ in range(n_polys):
        n = random.randint(MIN_VERTS, MAX_VERTS)
        cx, cy = random_point_in_allowed()
        angle = math.radians(random.choice(ALLOWED_ANGLES_DEG))
        length = random.uniform(80.0, 180.0)
        width = random.uniform(30.0, 70.0)
        pts = []
        for k in range(n):
            t = (k / (n - 1) - 0.5) * length
            off = (random.random() - 0.5) * width
            dx = t * math.cos(angle)
            dy = t * math.sin(angle)
            px = cx + dx - off * math.sin(angle)
            py = cy + dy + off * math.cos(angle)
            px, py = clamp_point_to_allowed(px, py)
            pts.append([px, py])
        polys.append({"points": pts})
    return polys


def deep_copy_polygons(polygons):
    return [
        {"points": [[float(x), float(y)] for x, y in poly.get("points", [])]}
        for poly in polygons
    ]


def mutate_polygons(base_polygons, jitter_xy=15.0, global_shift_xy=5.0, n_vertices=4):
    if not base_polygons:
        return random_polygons()
    polys = deep_copy_polygons(base_polygons)

    n_polys_to_shift = min(len(polys), random.randint(1, 2))
    for idx in random.sample(range(len(polys)), n_polys_to_shift):
        dx = random.uniform(-global_shift_xy, global_shift_xy)
        dy = random.uniform(-global_shift_xy, global_shift_xy)
        for p in polys[idx]["points"]:
            px, py = p
            px += dx
            py += dy
            p[0], p[1] = clamp_point_to_allowed(px, py)

    for poly in polys:
        if not poly["points"]:
            continue
        k = min(len(poly["points"]), n_vertices)
        for vidx in random.sample(range(len(poly["points"])), k):
            px, py = poly["points"][vidx]
            px += random.uniform(-jitter_xy, jitter_xy)
            py += random.uniform(-jitter_xy, jitter_xy)
            poly["points"][vidx] = list(clamp_point_to_allowed(px, py))

    return polys

In [ ]:
def fix_and_filter_polygons(polygons, smooth_r=3.0, min_area_frac=0.01, min_dist=4.0):
    fixed = []
    shapely_polys = []

    allowed_area_val = allowed_poly.area
    min_area = allowed_area_val * min_area_frac

    for poly in polygons:
        pts = [
            clamp_point_to_allowed(p[0], p[1])
            for p in poly.get("points", [])
            if isinstance(p, (list, tuple)) and len(p) == 2
        ]
        if len(pts) < 3:
            continue

        shp = SPolygon(pts)
        shp = shp.buffer(0)                        
        shp = shp.buffer(smooth_r).buffer(-smooth_r) 
        shp = shp.intersection(allowed_poly)
        if shp.is_empty:
            continue

        if isinstance(shp, (MultiPolygon, GeometryCollection)):
            polys_only = [g for g in shp.geoms if isinstance(g, SPolygon)]
            if not polys_only:
                continue
            shp = max(polys_only, key=lambda g: g.area)

        if not isinstance(shp, SPolygon):
            continue

        if (not shp.is_valid) or (shp.area < min_area):
            continue

        coords = list(shp.exterior.coords)[:-1]
        if len(coords) > MAX_VERTS:
            step = max(1, len(coords) // MAX_VERTS)
            coords = coords[::step]
        pts_new = [[float(x), float(y)] for x, y in coords]
        if len(pts_new) < 3:
            continue

        shapely_polys.append(shp)
        fixed.append({"points": pts_new})

    accepted = []
    accepted_shp = []
    for poly_dict, shp in sorted(zip(fixed, shapely_polys), key=lambda z: z[1].area, reverse=True):
        ok = True
        for shp_prev in accepted_shp:
            if shp.distance(shp_prev) < min_dist:
                ok = False
                break
        if ok:
            accepted.append(poly_dict)
            accepted_shp.append(shp)
        if len(accepted) >= MAX_POLYS:
            break

    if not accepted:
        return random_polygons()
    return accepted

In [6]:
def parse_llm_json(resp: str) -> dict:
    s = resp.strip()
    if s.startswith("```"):
        parts = s.split("```")
        if len(parts) >= 2:
            s = parts[1].strip()
        else:
            s = parts[0].strip()
    start = s.find("{")
    end = s.rfind("}")
    if start == -1 or end == -1 or start > end:
        raise ValueError("No JSON object found")
    return json.loads(s[start:end+1])

In [7]:
def save_polygons_figure(polygons, path):
    dx = X_MAX - X_MIN
    dy = Y_MAX - Y_MIN
    pad_x = 0.3 * dx
    pad_y = 0.1 * dy

    x_min = X_MIN - pad_x
    x_max = X_MAX + pad_x
    y_min = Y_MIN - pad_y
    y_max = Y_MAX + pad_y

    width = 900
    height = 600

    fig = go.Figure()

    for poly in polygons:
        pts = [
            p for p in poly.get("points", [])
            if isinstance(p, (list, tuple)) and len(p) == 2
        ]
        if len(pts) < 2:
            continue
        xs = [p[0] for p in pts] + [pts[0][0]]
        ys = [p[1] for p in pts] + [pts[0][1]]
        fig.add_trace(go.Scatter(
            x=xs,
            y=ys,
            mode="lines+markers",
            line=dict(width=2),
            marker=dict(size=5),
            showlegend=False,
        ))

    ax = [p[0] for p in allowed_area] + [allowed_area[0][0]]
    ay = [p[1] for p in allowed_area] + [allowed_area[0][1]]
    fig.add_trace(go.Scatter(
        x=ax,
        y=ay,
        mode="lines",
        line=dict(color="lightgray", dash="dash"),
        showlegend=False,
    ))

    fig.update_layout(
        xaxis_title="x (µm)",
        yaxis_title="y (µm)",
        xaxis=dict(range=[x_min, x_max]),
        yaxis=dict(range=[y_min, y_max]),
        width=width,
        height=height,
        plot_bgcolor="white",
    )

    fig.write_image(path, width=width, height=height, scale=2)

In [8]:
def shape_penalty(struct, max_aspect=8.0, gamma=0.05):
    aspects = []
    for poly in struct.polygons:
        xs = [p.x for p in poly.points]
        ys = [p.y for p in poly.points]
        w = max(xs) - min(xs)
        h = max(ys) - min(ys)
        if w <= 0 or h <= 0:
            continue
        aspect = max(w, h) / max(min(w, h), 1e-6)
        aspects.append(aspect)
    if not aspects:
        return 1.0
    mean_aspect = sum(aspects) / len(aspects)
    excess = max(0.0, mean_aspect - max_aspect)
    return 1.0 / (1.0 + gamma * excess)

In [9]:
def evaluate_polygons(polygons: list, iteration: int, run_dir: str):
    fixed_polygons = fix_and_filter_polygons(polygons)

    try:
        struct = polygons_to_structure(fixed_polygons)
    except Exception as e:
        print("polygons_to_structure failed:", e)
        struct = Structure(polygons=[])

    base_score = -comsol_objective(struct)
    shp_pen = shape_penalty(struct)
    final_score = base_score * shp_pen

    print(
        f"Iter {iteration}: base={base_score:.6f}, "
        f"shape_pen={shp_pen:.3f}, final={final_score:.6f}"
    )

    rec = {
        "iteration": iteration,
        "base_score": base_score,
        "shape_penalty": shp_pen,
        "final_score": final_score,
        "polygons": fixed_polygons,
    }
    with open(os.path.join(run_dir, f"{iteration:04d}.json"), "w", encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False, indent=2)

    png_path = os.path.join(run_dir, f"{iteration:04d}.png")
    save_polygons_figure(fixed_polygons, png_path)

    return final_score, fixed_polygons

In [10]:
task_description = (
    "You design polygonal obstacles inside a 2D microfluidic RBC trapping chamber.\n"
    f"The allowed design region is a quadrilateral with vertices {allowed_area} (in micrometers).\n\n"
    "The CFD simulator computes a scalar score that is higher when:\n"
    "- the total flow through 5 trapping slits is large relative to side+main flow,\n"
    "- all 5 slits have similar velocities,\n"
    "- curl and curvature of the flow are moderate (very large values are penalized).\n\n"
    "Your goal is to MAXIMIZE this score.\n\n"
    f"Design between {MIN_POLYS} and {MAX_POLYS} polygonal obstacles. Each obstacle must be an elongated, wall-like shape\n"
    "that guides the flow toward the trapping slits, not an isolated circular blob.\n"
    "Avoid using a single long straight bar; instead, prefer 2–4 walls composed of several straight segments\n"
    "at standard angles (0, 30, 45, 60, 90, 120, 135 degrees), forming gently curved or broken-line patterns\n"
    "similar to the existing trapping structures on the right side of the chamber.\n"
    "All polygons must be simple (no self-intersections) and entirely inside the allowed region. "
    "Distinct polygons must not overlap or touch each other.\n\n"
    "Return ONLY valid JSON with structure:\n"
    "{\n"
    "  \"polygons\": [\n"
    "    {\"points\": [[x1, y1], [x2, y2], ..., [xk, yk]]},\n"
    f"    ... (between {MIN_POLYS} and {MAX_POLYS} polygons)\n"
    "  ]\n"
    "}\n"
    f"Coordinates in micrometers, x in [{X_MIN:.1f}, {X_MAX:.1f}], y in [{Y_MIN:.1f}, {Y_MAX:.1f}].\n"
    f"Each polygon should have between {MIN_VERTS} and {MAX_VERTS} vertices.\n"
)

In [11]:
def sample_from_llm(best, history, explore_strength: float) -> list:
    top = sorted(history, key=lambda h: h["score"], reverse=True)[:3]
    prompt = {
        "task": task_description,
        "current_best": {"score": best["score"], "polygons": best["polygons"]},
        "recent_solutions": [
            {"score": h["score"], "polygons": h["polygons"]} for h in top
        ],
        "mode": "exploration" if explore_strength > 0.5 else "exploitation",
        "instruction": (
            "Propose a NEW configuration by modifying or combining the best and recent solutions. "
            "Apply stronger changes when mode == exploration (move several vertices, change angles, "
            "add gentle bends). Ensure all polygons are simple (no self-intersections), inside the "
            "allowed region, and well separated (no overlaps or touching). Return ONLY JSON."
        ),
    }
    msg = json.dumps(prompt, ensure_ascii=False)
    resp = llm.draw_sample(msg)

    try:
        data = parse_llm_json(resp)
        polygons = data.get("polygons", [])
        if not isinstance(polygons, list) or not polygons:
            raise ValueError("No polygons in JSON")
    except Exception as e:
        print("LLM JSON parse error, fallback to mutate best:", e)
        print("Raw head:", resp[:200])
        polygons = deep_copy_polygons(best["polygons"]) if best["polygons"] else random_polygons()

    jitter = 10.0 + 30.0 * explore_strength
    shift = 5.0 + 15.0 * explore_strength
    polygons = mutate_polygons(polygons, jitter_xy=jitter, global_shift_xy=shift, n_vertices=4)
    return polygons

In [ ]:
run_name = datetime.now().strftime("llm_%Y-%m-%d_%H-%M-%S")
run_dir = os.path.join("logs", run_name)
os.makedirs(run_dir, exist_ok=True)
print("Run dir:", run_dir)


def run_llm_optimization(n_iters=40, max_stagnation=8):
    history = []

    print("Evaluating baseline (no obstacles)...")
    baseline_struct = Structure(polygons=[])
    baseline_score = -comsol_objective(baseline_struct)
    print("Baseline score:", baseline_score)

    best = {"polygons": [], "score": baseline_score}
    history.append(best)

    best_scores = []
    stagnation = 0

    for i in range(1, n_iters + 1):
        print(f"\nITERATION {i}")

        explore_strength = min(1.0, stagnation / max(max_stagnation, 1))
        if i <= 3:
            raw_polygons = random_polygons()
            print("Using random initialization")
        else:
            raw_polygons = sample_from_llm(best, history, explore_strength)
            print(f"Using LLM+mutation, explore_strength={explore_strength:.2f}")

        score, fixed_polygons = evaluate_polygons(raw_polygons, iteration=i, run_dir=run_dir)

        history.append({"polygons": fixed_polygons, "score": score})

        if score > best["score"] + 1e-6:
            best = {"polygons": fixed_polygons, "score": score}
            stagnation = 0
        else:
            stagnation += 1

        best_scores.append(best["score"])
        print(f"Current best: {best['score']:.6f}, stagnation={stagnation}")

    iters = list(range(1, len(best_scores) + 1))
    fig = go.Figure(go.Scatter(x=iters, y=best_scores, mode="lines+markers"))
    fig.update_layout(
        title=f"Convergence {run_name}",
        xaxis_title="Iteration",
        yaxis_title="Best score",
    )
    conv_png = os.path.join(run_dir, "convergence.png")
    fig.write_image(conv_png, width=800, height=600, scale=2)
    print("Convergence PNG saved to", conv_png)

    png_files = sorted(glob.glob(os.path.join(run_dir, "*.png")))
    png_files = [p for p in png_files if os.path.basename(p)[:4].isdigit()]
    if png_files:
        images = [imageio.imread(p) for p in png_files]
        gif_path = os.path.join(run_dir, "evolution.gif")
        imageio.mimsave(gif_path, images, duration=0.8)
        print("GIF saved to", gif_path)

    return baseline_score, best, history, best_scores

Run dir: logs\llm_2025-12-10_03-32-44


In [ ]:
baseline_score, best_solution, history, best_scores = run_llm_optimization(
    n_iters=100,
    max_stagnation=6,
)
print("\nFINAL BEST SCORE:", best_solution["score"])
print("Run dir:", run_dir)

Evaluating baseline (no obstacles)...
Start COMSOL
score: 0.159854 vlcts: [0.000382, 0.000486, 0.000328, 0.000277, 0.000228, 0.001271, 0.001684] curl: 951.79 curv: 3.36e+06
Baseline score: 0.15985385562881155

ITERATION 1
Using random initialization
Start COMSOL
score: 0.183901 vlcts: [0.000376, 0.000479, 0.000322, 0.000276, 0.000237, 0.001257, 0.001693] curl: 1806.43 curv: 5.79e+07
Iter 1: base=0.183901, shape_pen=1.000, final=0.183901
Current best: 0.183901, stagnation=0

ITERATION 2
Using random initialization
Start COMSOL
score: 0.176915 vlcts: [0.00042, 0.000535, 0.000362, 0.000296, 0.000207, 0.001399, 0.001538] curl: 2303.85 curv: 3.68e+07
Iter 2: base=0.176915, shape_pen=1.000, final=0.176915
Current best: 0.183901, stagnation=1

ITERATION 3
Using random initialization
Start COMSOL
score: 0.20863 vlcts: [0.000282, 0.000373, 0.000279, 0.00033, 0.000283, 0.000897, 0.002087] curl: 2729.25 curv: 9.65e+07
Iter 3: base=0.208630, shape_pen=1.000, final=0.208630
Current best: 0.208630, 

In [19]:
best_solution

{'polygons': [{'points': [[15.004010023829677, 66.05155639098152],
    [15.05225400886718, 66.11140445892181],
    [47.30302046937512, 102.10251813255827],
    [47.35277411500424, 102.15286605578636],
    [47.4185002428851, 102.12659031144761],
    [134.2377719900938, 62.400605105468166],
    [134.29926995902426, 62.368638674682636],
    [134.2442261258129, 62.32651944742013],
    [65.56088673022748, 15.041917077034617],
    [65.49937378069437, 15.003892920508843],
    [65.42903933912442, 15.020707966414504],
    [15.071283687165112, 29.71626885724364],
    [15.00354657825809, 29.739730624264606],
    [15.0, 29.811328055904866],
    [15.0, 65.97478925584319]]},
  {'points': [[25.07381936901824, 259.9876967718303],
    [37.03960403829046, 258.1186176492897],
    [38.12809533086704, 258.4730197310961],
    [39.004595720954406, 259.20933569301764],
    [39.54148537568882, 260.22035677501486],
    [39.660592318447826, 261.3588767202905],
    [40.0031572930823, 289.9982505293986],
    [66.7

In [14]:
BASE_MODEL_PATH = r"Data\Prepared_NSS_0_10um_frog_RBC_2025.mph"  
OUTPUT_MODEL_PATH = os.path.join(run_dir, r"optimized_trap_llm_best.mph")

In [ ]:
def export_best_to_comsol(best_solution, base_model_path, output_model_path, client):
    raw_polygons = best_solution.get("polygons", [])
    polygons = fix_and_filter_polygons(raw_polygons)

    print("Exporting", len(polygons), "polygons to COMSOL")

    model = client.load(base_model_path)
    geom = model.java.component("comp1").geom("geom1")

    try:
        tags = list(geom.features().tags())
        for tag in tags:
            if str(tag).startswith("pol"):
                geom.feature(tag).remove()
    except Exception as e:
        print("Warning: failed to clear old pol* features:", e)

    for i, poly in enumerate(polygons):
        name = f"pol{i+1}"
        pts = poly.get("points", [])
        if len(pts) < 3:
            continue
        xs = " ".join(str(p[0]) for p in pts)
        ys = " ".join(str(p[1]) for p in pts)

        try:
            geom.create(name, "Polygon")
        except Exception:
            pass
        feat = geom.feature(name)
        feat.set("x", xs)
        feat.set("y", ys)

    try:
        geom.run()
    except Exception as e:
        print("Geometry run error during export:", e)

    model.save(output_model_path)
    print(f"Saved optimized model to: {output_model_path}")
    return model

In [16]:
_ = export_best_to_comsol(
    best_solution=best_solution,
    base_model_path=BASE_MODEL_PATH,
    output_model_path=OUTPUT_MODEL_PATH,
    client=comsol_client,
)

Exporting 3 polygons to COMSOL
Saved optimized model to: logs\llm_2025-12-10_03-32-44\optimized_trap_llm_best.mph
